In [ ]:
# ==============================================================================
# Hito 2: Feature Importance - Modelo Multiclase (Ataques)
# Estrategia: Undersampling + CTGAN (Oversampling) + XGBoost + SHAP
# ==============================================================================

import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import gc
import collections
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from imblearn.under_sampling import RandomUnderSampler
from ctgan import CTGANSynthesizer

# Inicializar JS para gráficos de SHAP y configurar estilo visual
shap.initjs()
plt.style.use('seaborn-v0_8-whitegrid')

# Verificar si PyTorch detecta la GPU antes de empezar
print(f"GPU detectada por PyTorch: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")

In [ ]:
print("Cargando dataset...")
df = pd.read_feather('dataset_eda_temp.feather')

# 1. Eliminar columnas con casi 100% ceros (INPLACE para no duplicar RAM)
cols_to_drop = [
    'telnet', 'smtp', 'irc', 'cwr_flag_number', 'ece_flag_number', 
    'dhcp', 'ssh', 'drate', 'arp', 'dns', 'http', 'https'
]
cols_existentes = [col for col in cols_to_drop if col in df.columns]
df.drop(columns=cols_existentes, inplace=True)
gc.collect()

# 2. Quedarnos SOLO con el tráfico malicioso
print("Filtrando tráfico benigno...")
df = df[df['label'] != 'BenignTraffic']

# Forzar limpieza de memoria tras el filtrado
gc.collect()
print(f"Dataset de solo ataques listo. Forma actual: {df.shape}")

In [ ]:
print("Iniciando pipeline de balanceo híbrido con CTGAN...")

# 1. Separar X e y temporalmente
X_raw = df.drop(columns=['label'])
y_text = df['label']

# 2. Definir los límites de balanceo
MAX_SAMPLES = 50000  # Techo: Para no saturar RAM/Tiempo
MIN_SAMPLES = 15000  # Piso: Mínimo necesario para XGBoost

conteo_original = y_text.value_counts().to_dict()
dict_under = {clase: min(cantidad, MAX_SAMPLES) for clase, cantidad in conteo_original.items()}
dict_over = {clase: max(dict_under[clase], MIN_SAMPLES) for clase in dict_under}

print(f"Limpiando RAM principal antes del remuestreo...")
del df 
gc.collect()

# 3. FASE 1: Bajar el techo (Random UnderSampler)
print(f"1/2 - Aplicando Undersampling a clases mayoritarias (Máx {MAX_SAMPLES})...")
rus = RandomUnderSampler(sampling_strategy=dict_under, random_state=42)
X_under, y_under = rus.fit_resample(X_raw, y_text)

del X_raw
gc.collect()

# Reconstruimos DataFrame temporal para CTGAN
df_balanced = pd.concat([X_under, y_under], axis=1)

# 4. FASE 2: Subir el piso con CTGAN
print(f"2/2 - Generando datos sintéticos con CTGAN (Mín {MIN_SAMPLES})...")

# Definir variables discretas/categóricas para no generar "paquetes imposibles"
columnas_discretas = [
    'ack_flag_number', 'psh_flag_number', 'syn_flag_number', 
    'rst_flag_number', 'fin_flag_number', 'urg_flag_number'
]
columnas_discretas = [col for col in columnas_discretas if col in X_under.columns]

for clase, target_count in dict_over.items():
    current_count = dict_under[clase]
    num_a_generar = target_count - current_count
    
    if num_a_generar > 0:
        print(f"  -> Entrenando CTGAN para: {clase} (Generando {num_a_generar} sintéticos)")
        
        # Aislar los datos reales
        df_clase_real = df_balanced[df_balanced['label'] == clase].drop(columns=['label'])
        
        # Entrenar modelo CTGAN (epochs=50 para EDA; usar 300 para modelo final TCN)
        ctgan = CTGANSynthesizer(epochs=50, verbose=False, cuda=True) 
        ctgan.fit(df_clase_real, discrete_columns=columnas_discretas)
        
        # Generar paquetes sintéticos
        datos_sinteticos = ctgan.sample(num_a_generar)
        datos_sinteticos['label'] = clase 
        
        # Concatenar al dataset principal
        df_balanced = pd.concat([df_balanced, datos_sinteticos], ignore_index=True)
        
        # ----------------------------------------------------
        # PROTOCOLO ESTRICTO DE LIMPIEZA DE MEMORIA VRAM/RAM
        # ----------------------------------------------------
        del ctgan               
        del df_clase_real       
        del datos_sinteticos
        gc.collect()            
        torch.cuda.empty_cache()
        # ----------------------------------------------------

# 5. Preparación final
X_bal = df_balanced.drop(columns=['label'])
y_bal_text = df_balanced['label']

le = LabelEncoder()
y_final = le.fit_transform(y_bal_text)
nombres_clases = le.classes_

print(f"\nForma final de los datos balanceados: {X_bal.shape}")
X = X_bal
y = y_final

In [ ]:
# Split 80/20 estratificado
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Entrenando clasificador proxy XGBoost Multiclase...")

clf_multi = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist',
    objective='multi:softprob', 
    num_class=len(nombres_clases),
    n_jobs=-1,
    random_state=42
)

clf_multi.fit(X_train, y_train)

# Evaluación rápida de sanidad
y_pred = clf_multi.predict(X_test)
print("\n--- Desempeño del Modelo Proxy ---")
print(f"Accuracy Global: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-Score Macro: {f1_score(y_test, y_pred, average='macro'):.4f}")

# Guardar en JSON (Estándar de XGBoost, seguro para cruzar OS Windows/Linux)
model_path = "xgb_modelo_multiclase_balanceado.json"
clf_multi.save_model(model_path)
print(f"Modelo proxy guardado en: {model_path}")

In [ ]:
# Tomamos una muestra de 10,000 para SHAP. En multiclase, evaluar 
# 33 clases simultáneamente es muy denso visual y computacionalmente.
X_shap = X_train.sample(n=10000, random_state=42)

print("Calculando valores SHAP multiclase (esto tomará unos minutos)...")
explainer = shap.TreeExplainer(clf_multi)
shap_values = explainer.shap_values(X_shap)
print("Cálculo finalizado. Generando visualización...")

# Renderizar el Summary Plot (Barplot Apilado)
plt.figure(figsize=(14, 10))
plt.title("Top 20 Features - Importancia para Clasificación Fina de Ataques", fontsize=16)

shap.summary_plot(
    shap_values, 
    X_shap, 
    class_names=nombres_clases, 
    plot_type="bar", 
    max_display=20,
    show=False 
)

# Ajuste fino de la leyenda para evitar que tape el gráfico
plt.legend(loc='center left', bbox_to_anchor=(1.05, 0.5), fontsize=9, ncol=2, title="Clases de Ataques")
plt.tight_layout()
plt.show()